### Regression benchmark template (have data optimization/processing in a separate file)

**See the "modules_to_install.txt" for needed pip installations.**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import tensorflow as tf
import keras
from keras import layers


### Phase 1: Load the data

**To keep this benchmarking notebook more simple, have data processing/optimization etc. in a separate notebook, and save the final optimized version into a new csv-file which you will use here.**

In [2]:
# load the diamonds dataset, replace this with your own .csv -data
# this version of data should be READY TO GO to training
# which means => only numeric values in the dataset
df = pd.read_csv("processed_regression_diamonds.csv")
df.head()

,carat,cut,color,clarity,table,price,x,y,z
0,0.29,4,2,4,58.0,334,4.20,4.23,2.63
1,0.26,3,3,3,55.0,337,4.07,4.11,2.53
2,0.30,2,1,3,55.0,339,4.25,4.28,2.73
3,0.31,5,1,2,54.0,344,4.35,4.37,2.71
4,0.32,4,6,1,58.0,345,4.38,4.42,2.68


### X/y + train/test -splits + other data setups

**Fix the categoricals, continuous and target variable according to your own dataset.**

In [3]:
# CatBoost requires we save our continuous and categorical variables separately into lists
categorical_variables = ["cut", "color", "clarity"]

# continuous variables also into a list
continuous_variables = ["carat", "table", "price", "depth", "x", "y", "z"]

# the usual X/y -split
X = df.drop("price", axis=1)
y = df['price']

# usual train/test -split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# NOTE! SCALING => some of the algorithms require this
scaler = StandardScaler()

# create separate versions for the scaled data
# because we need both unscaled and scaled versions later
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### List all our models that we want to benchmark

In [4]:
# define our model test dictionary

# we have quite a lot of data ~50000 rows
# keep in mind => this amount of data might be too heavy for KNN

# for the hyperparameter optimization results, see
# "regression_hyperparameter_optimization.ipynb"

# RandomizedSearchCV results LightGBM:
# {'learning_rate': np.float64(0.12362068965517242), 'max_depth': 25, 'num_leaves': 47}

models = {
    "Linear Regression": LinearRegression(),
    "SVM": SVR(),
    "KNN": KNeighborsRegressor(),
    "Random Forest": RandomForestRegressor(),
    "XGBoost": xgb.XGBRegressor(enable_categorical=True, objective="reg:squarederror"),
    "CatBoost": cb.CatBoostRegressor(verbose=0),
    'LightGBM-default': lgb.LGBMRegressor(objective='regression'),
    'LightGBM-opt': lgb.LGBMRegressor(objective='regression', learning_rate=0.12362068965517242, max_depth=25, num_leaves=47),

}

### We need a for-loop to train all the models in one go + gather metrics

In [5]:
# our benchmarking code!

# this will contain our results later after the benchmarks
results = []

# loop through the models
for name, model in models.items():
    # we have to react to certain algorithms
    # because they have special requirements for fit()

    print("Starting..." + name)

    # SVM / KNN require scaled data
    if name in ["SVM", "KNN"]:
        model.fit(X_train_scaled, y_train)
        predictions = model.predict(X_test_scaled)
    elif name == "CatBoost":
        # CatBoost requires we specify which columns are categories
        # this includes both ordinal and nominal categories
        model.fit(X_train, y_train, cat_features=categorical_variables)
        predictions = model.predict(X_test)
    else:
        # all the other algorithms
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

    # after training this particular algorithm, gather the metrics
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    # save the metrics for this model into the results
    results.append([name, mae, mse, rmse, r2])

Starting...Linear Regression
Starting...SVM
Starting...KNN
Starting...Random Forest
Starting...XGBoost
Starting...CatBoost
Starting...LightGBM-default
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000167 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 937
[LightGBM] [Info] Number of data points in the train set: 33375, number of used features: 8
[LightGBM] [Info] Start training from score 2268.119341
Starting...LightGBM-opt
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000137 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 937
[LightGBM] [Info] Number of data points in the train set: 33375, number of used features: 8
[LightGBM] [Info] Start training from score 2268.119341


### Let's visualize the results

**This code was originally created with the help of ChatGPT**

In [6]:
metrics_df = pd.DataFrame(results, columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2'])

# change these highlight colors as you wish
# default style is dart theme / mode, modify
# variable default_highlight_style if you wish 
# to use light theme instead
light_theme = 'background: yellow'
dark_theme = 'background: #8C670A'

# you can alternate the highlight styling based on your theme
default_highlight_style = dark_theme

# helper function that highlights the best model of each metric
def highlight_best_metrics(row):
    # default styles for everything is empty in the beginning
    styles = ['' for _ in row]

    # index 1 => MAE (index 0 => model name)
    if row['MAE'] == metrics_df['MAE'].min():
        styles[1] = default_highlight_style

    # index 2 = > MSE
    if row['MSE'] == metrics_df['MSE'].min():
        styles[2] = default_highlight_style

    # index 3 = > RMSE
    if row['RMSE'] == metrics_df['RMSE'].min():
        styles[3] = default_highlight_style

    # index 4 = > R-squared (R2)
    if row['R2'] == metrics_df['R2'].max():
        styles[4] = default_highlight_style

    return styles


# apply the custom styles based on the min/max metrics
highlight_df = metrics_df.style.apply(highlight_best_metrics, axis=1)


highlight_df

,Model,MAE,MSE,RMSE,R2
0,Linear Regression,299.092446,187412.167166,432.911269,0.937125
1,SVM,374.842911,371950.237118,609.877231,0.875213
2,KNN,172.735403,76082.352900,275.830297,0.974475
3,Random Forest,139.523457,56741.346928,238.204423,0.980964
4,XGBoost,141.883743,55611.125000,235.820112,0.981343
5,CatBoost,141.716469,53757.017982,231.855597,0.981965
6,LightGBM-default,144.256232,54874.984104,234.254102,0.981590
7,LightGBM-opt,139.753633,53370.200423,231.019913,0.982095


In [7]:
# results before optimizing dataset with RANSAC + MAD + Isolation Forest:
# for the optimization steps, see "regression_data_processing.ipynb"

# 0	Linear Regression	815.950987	1508538.713676	1228.225840	0.906935
# 1	SVM	                1293.134118	7579792.525461	2753.142300	0.532385
# 2	KNN	                380.828532	520399.930615	721.387504	0.967895
# 3	Random Forest	    271.793133	302519.512836	550.017739	0.981337
# 4	XGBoost	            278.312844	304370.199140	551.697561	0.981223
# 5	CatBoost	        280.096492	293394.766881	541.659272	0.981900
# 6	LightGBM	        283.569610	295962.867879	544.024694	0.981741

# Note: since the distribution of the dataset has been altered during the optimizations
# the metrics MAE, MSE and RMSE are no long comparable

# typically when you remove a huge amount of outliers => metrics improve

# keep an eye on the R-squared, it's comparabable even after changing 
# the distributions during the optimization